# 02 — Build SILVER & GOLD Layers

Creates Dynamic Tables (SILVER) and analytics views (GOLD).


In [ ]:
-- Build SILVER & GOLD in a single transaction
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE WAREHOUSE HYDRAB_HOL_WH;

-- SILVER: Vehicles from Salesforce Asset
CREATE OR REPLACE DYNAMIC TABLE SILVER.VEHICLES_SILVER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = HYDRAB_HOL_WH
AS
SELECT
  "Chassis_Number__c" AS vin,
  "Name" AS model,
  "AccountId" AS customer_id,
  "Bus_Depot__c" AS depot_id,
  "Status" AS status,
  "Production_Status__c" AS production_status,
  "Delivery_Status__c" AS delivery_status,
  "Delivery_Date__c" AS delivery_date,
  "Registration__c" AS registration,
  "LatestLocation__Latitude__s" AS last_latitude,
  "LatestLocation__Longitude__s" AS last_longitude,
  "LatestMileage__c" AS last_mileage
FROM BRONZE.ASSET
WHERE "Chassis_Number__c" IS NOT NULL;

In [ ]:
-- SILVER: Telemetry from Odos (flatten signals array)
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE WAREHOUSE HYDRAB_HOL_WH;

CREATE OR REPLACE DYNAMIC TABLE SILVER.TELEMETRY_SILVER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = HYDRAB_HOL_WH
AS
WITH parsed AS (
  SELECT
    PARSE_JSON(RAW_JSON):vin::STRING AS vin,
    PARSE_JSON(RAW_JSON):startTime::TIMESTAMP AS event_time,
    PARSE_JSON(RAW_JSON):vehicleCustomer::STRING AS customer_name,
    sig.value:name::STRING AS signal_name,
    sig.value:values[0]:value::STRING AS reading_value
  FROM BRONZE.ODOS_EVENTS,
    LATERAL FLATTEN(input => PARSE_JSON(RAW_JSON):signals) sig
)
SELECT
  vin,
  event_time,
  customer_name,
  MAX(CASE WHEN signal_name = 'LOCATION' THEN SPLIT_PART(reading_value, ';', 2)::FLOAT END) AS latitude,
  MAX(CASE WHEN signal_name = 'LOCATION' THEN SPLIT_PART(reading_value, ';', 1)::FLOAT END) AS longitude,
  MAX(CASE WHEN signal_name = 'MBMSStat1_DisplayedSOC_18FFB5F3_2' THEN reading_value::FLOAT END) AS battery_soc,
  MAX(CASE WHEN signal_name ILIKE '%VehicleSpeed%' THEN reading_value::FLOAT END) AS speed_kmh,
  MAX(CASE WHEN signal_name ILIKE '%TotalVehDistance%' THEN reading_value::FLOAT END) AS odometer_km
FROM parsed
GROUP BY vin, event_time, customer_name;

In [ ]:
-- SILVER: Defect events
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE WAREHOUSE HYDRAB_HOL_WH;

CREATE OR REPLACE DYNAMIC TABLE SILVER.DEFECTS_SILVER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = HYDRAB_HOL_WH
AS
SELECT
  "Defect__c" AS defect_id,
  "Name" AS event_name,
  "Type__c" AS event_type,
  "Body__c" AS description,
  "Actor__c" AS actor,
  "Old_Value__c" AS old_value,
  "New_Value__c" AS new_value,
  "CreatedDate" AS created_date
FROM BRONZE.DEFECT_EVENT
WHERE "Defect__c" IS NOT NULL;

In [ ]:
-- GOLD views
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA GOLD;

CREATE OR REPLACE VIEW DIM_VEHICLE AS
SELECT * FROM SILVER.VEHICLES_SILVER;

CREATE OR REPLACE VIEW FCT_LATEST_TELEMETRY AS
SELECT *
FROM SILVER.TELEMETRY_SILVER
QUALIFY ROW_NUMBER() OVER (PARTITION BY vin ORDER BY event_time DESC) = 1;

CREATE OR REPLACE VIEW FCT_DEFECT AS
SELECT * FROM SILVER.DEFECTS_SILVER;

## Done!
SILVER Dynamic Tables and GOLD views created.
